In [13]:
from typing import Annotated
from dotenv import load_dotenv

In [14]:
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph
from langgraph.graph import START
from langgraph.graph import END
from langgraph.graph.message import add_messages

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [16]:
load_dotenv()

True

In [17]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {'messages':[llm.invoke(state['messages'])]}

builder = StateGraph(State)
builder.add_node('chatbot_node',chatbot)

builder.add_edge(START, 'chatbot_node')
builder.add_edge('chatbot_node', END)

graph = builder.compile()

In [19]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}

In [20]:
response = graph.invoke({"messages":[message]})

In [21]:
response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='e30310bf-fee5-4cbd-8b04-d93cfa056e3e'),
 AIMessage(content='Neil Armstrong', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--0efedaf2-8bc3-44bd-b964-d86d36a7eb28-0', usage_metadata={'input_tokens': 14, 'output_tokens': 3, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})]

In [22]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: In football (soccer), "GOAT" stands for **Greatest Of All Time**. It's a common term used to describe the player considered to be the best ever.

Currently, the two most frequently debated and considered GOATs in football are:

*   **Lionel Messi:** An Argentinian forward who has played most of his career for FC Barcelona and currently plays for Inter Miami. He has won numerous Ballon d'Or awards (more than any other player), Champions League titles, and league titles. He also captained Argentina to victory in the 2022 FIFA World Cup.

*   **Cristiano Ronaldo:** A Portuguese forward who has played for clubs like Manchester United, Real Madrid, Juventus, and currently plays for Al-Nassr. He has also won multiple Ballon d'Or awards, Champions League titles, and league titles. He is the all-time top scorer in men's international football.

The GOAT debate between Messi and Ronaldo is a long-standing one, with fans often arguing for their preferred player based on various factors like